# Where Does the Juice Leak? Following Combine movement into NFL games

Scouts say a player has **juice** when he moves with an extra gear. The Combine tries to measure it in the
"Underwear Olympics": a 40, a few cone drills and position work, all run in shorts with no one across from you.
Now every rep at the Combine carries chip tracking, and so does every NFL snap. That lets us measure the
**same movement traits on both sides** and follow them link by link:

> **Combine trait → same trait in NFL games → football result**

Where the chain holds, tracking tells a team something useful. Where it breaks, we learn where the juice leaks.
Our answer in one line: **you can get speed from a stopwatch, but change of direction is where tracking earns
its keep, mainly for wide receivers and defensive backs.**

*Data: every 2023–2025 rookie with Combine tracking in the five competition position groups (DL split into
interior DL and EDGE), matched to regular-season tracking. Every number below comes from a single pipeline run;
its ID is printed in the next cell, and the appendix rechecks every quoted number against that run's tables.*

In [ ]:
# Setup. Runs the pipeline if this run's outputs are not already present (about 20 minutes the first time).
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "programs" / "juice.py").exists())
os.chdir(ROOT)
OUT = Path("/kaggle/working/juice_nb_output") if Path("/kaggle/working").is_dir() else ROOT / "juice_nb_output"
STEPS = ["juice.py", "juice_physics.py", "juice_transfer.py", "juice_predictive.py", "juice_casework.py", "juice_figures.py"]
if not (OUT / "secondary_checks.csv").exists():
    for s in STEPS:
        subprocess.run([sys.executable, str(ROOT / "programs" / s)], check=True)
RUN = json.loads((OUT / "run_manifest.json").read_text())
T = lambda name: pd.read_csv(OUT / name)
FIG = lambda name: display(Image(filename=str(OUT / "figures" / name), width=900))
print("Run", RUN["run_id"], "| players:", RUN["counts"])

## 1. Measuring the same thing twice

**Combine side.** Each player gets three tracked pieces, each scored like baseball's OPS+ (100 = position
average, adjusted for body size):

| Piece | Combine source | Game version |
|---|---|---|
| **Speed** | fastest 10 yards of the tracked 40 | 95th-percentile top speed per play |
| **Burst** | time from yard 2 to yard 10 of the tracked 40 | peak acceleration out of a stop |
| **Agility** | position drills (bag, route, back-pedal), used only where a group's drills agree (alpha ≥ 0.5) | hardest change of direction per play |

**Game side.** Each regular-season play is adjusted for context (alignment, route and play type). A player
needs 50 usable plays to get a game score, and each piece is pulled toward average in proportion to its
reliability. We report the pieces separately because they don't add up to one trait in games (the game
composite reaches alpha 0.5 only for WR and OL).

**One fix mattered.** The obvious burst number (distance covered in the first 1.5 seconds) correlates only
**0.20** with a player's official 40, because the clock starts on a single noisy frame. Timing yards 2 to 10
raises that to **0.49**. It is still the weakest piece: on a second 40 attempt, a player's burst repeats at
only **0.47**, against **0.91** for top speed.

**Same physics on both sides.** As an independent check, we fit an *acceleration–speed profile*, a method
from soccer GPS research, to every frame a player logs. It needs no start signal and no play context, so the
Combine and games produce the same two numbers: first-step power (a0) and top gear (v0). Combine top gear
predicts game top gear at **0.22 [0.13, 0.32]** (n = 392, pre-registered test P5).

In [ ]:
FIG("fig2_asp_overlay.png")

## 2. What tracking adds beyond the stopwatch

Teams already have the 40. The question that matters is what the chips add **on top of it**. For each tracked
piece, we hold the official 40 fixed and ask whether the piece still predicts the same trait in games:

In [ ]:
FIG("fig3_tracking_adds.png")

- **Speed adds nothing.** The tracked fastest-10 and the official 40 correlate 0.89. They measure the same
  thing.
- **Burst adds nothing yet**, even after the fix (+0.10 [−0.02, +0.21]).
- **Agility adds real signal:** +0.17 [+0.03, +0.30] holding the 40 *and* a RAS-style composite of every
  official test (n = 198, pre-registered test **P1**). It survives adding draft pick and snaps (+0.17), and it
  is strongest for **WR and DB** (+0.30 [+0.15, +0.44], n = 125).

Agility is the one piece a stopwatch can't see, because no official drill isolates position-specific change
of direction. It is also the piece the old Combine scores leave out.

**What we tried and couldn't show.** We pre-registered that tracked burst would predict NFL get-off for pass
rushers (P2). It came in at +0.15 [−0.09, +0.38] (n = 68), well below the 40's +0.37 on the same players. On
this data, pass-rush first step is still a stopwatch story.

**Head-to-head with existing scores (exploratory).** On game movement, Combine Juice+ carries over at 0.27,
against 0.24 for the 40, 0.14 for RAS-style and 0.16 for NGS athleticism (n = 249). That edge comes from
removing body size: **a size-adjusted 40 reaches 0.29**, level with Juice+. Out of class, Juice+ beats the
plain 40 by +0.02 R² [−0.03, +0.06], which is not a clear win. So we don't claim to "beat the 40." We claim the narrower
point above: the agility piece adds what the 40 can't.

## 3. Following the chain into football results

Moving well in games only matters if it changes results. For each group we test both links, using a play-level
outcome a coach would recognize: route-adjusted separation at the throw (WR, TE), alignment-adjusted get-off and
pressure rate (DL, EDGE), peak pressure probability allowed (OL) and coverage EPA (DB).

In [ ]:
FIG("fig1_chain.png")

**Link 1 (Combine → games) mostly holds.** WR speed (0.57), WR agility (0.34) and DB agility (0.37) all carry
into games with intervals above zero.

**Link 2 (games → results) is where the juice leaks.** Across groups, moving well in games has only a weak tie
to the result stats. Some results are noisy too: coverage EPA repeats at only 0.44 within a player
(split-half), against 0.78 for WR separation. A **mixed model** puts the two together:
each play's outcome = context + player effect, and the player effect = β × Combine trait. Here, "transfer" is
β divided by the spread in true player ability, which works like a correlation. Tracked agility transfers to
game agility for DB (**+0.41 [+0.19, +0.63]**) and WR (**+0.33 [+0.08, +0.58]**). For WR it does **not**
transfer to separation at the throw (pre-registered **P3**: −0.05 [−0.32, +0.21]).

**Size is the hidden variable in results.** Within a position, lighter players separate more (WR separation
vs weight: −0.42) and get off faster (DL −0.43). Raw athleticism scores partly reward being light, so on raw
results Juice+, which removes size by design, trails NGS, RAS-style and the 40. When we take weight and height
out of **every** score, the gaps close. Pooled over every position-and-result pair, Juice+ minus each score is
−0.05 for NGS, −0.04 for RAS-style and −0.01 for the 40, and every interval crosses zero. **No athleticism
score, ours included, predicts football results well**, with correlations around +0.03 to +0.08. That is the
honest ceiling for Combine movement data at this sample size.

## 4. Lead story: agility for receivers and defensive backs

Everything above points to one trait and two positions. For WR and DB:

1. **It carries into games, beyond the 40.** Tracked drill agility predicts game agility with the 40 held
   fixed: **+0.33 [+0.15, +0.48]** (n = 125). It survives adding RAS-style, draft pick, play count and body
   size (+0.33 to +0.35), and a within-position shuffle of the agility scores never gets close
   (permutation p = 0.001).
2. **It is specific.** Drill agility predicts game agility, not game top speed or burst; the tracked
   fastest-10 predicts game top speed instead. Each Combine number finds its own trait.
3. **It isn't a selection effect.** Players who never reach 50 plays are mostly late picks, not low-agility
   players (Combine agility percentile 0.49 vs 0.51). Re-weighting for who reached the field (+0.34), and
   rebuilding game agility at floors from 10 to 200 plays (+0.25 to +0.34), leave it in place.
4. **It predicts growth.** Holding a player's rookie-year game agility fixed, his Combine agility still
   predicts his **year-2** game agility: **+0.34 [+0.12, +0.54]** (n = 63). The 40 predicts nothing there
   (+0.01). The 16 WR and DB whose drill agility ran a full SD ahead of their rookie game agility gained
   +0.71 SD the next year; everyone else slipped −0.48.

What it does **not** do: agility doesn't clearly beat the 40 head-to-head at predicting game agility (each
holding the other, the gap is +0.19 [−0.09, +0.43]), and in this sample game agility doesn't yet show up in
separation or coverage EPA. It is a movement signal with a development payoff, not a results model.

The prospect a stopwatch misses looks like this:


In [ ]:
FIG("fig6_prospect_card.png")

**Malik Heath** (WR, undrafted, 2023) ran a 4.64 40, a Speed score of 86, and his overall Combine Juice+ was a
below-average 91. A 40-first board passes on him. His tracked drill agility was 119, among the best of any
late-round WR or DB, and in 229 NFL plays his game agility came in above average at 106. *(Chosen as the late-round
WR or DB with the highest tracked agility and 100+ plays. This is a case study, not a test: across every pair of
same-class WRs or DBs with matching 40s and very different drill agility, the more agile player moved better
in games only 53% of the time. The signal is real for groups and too weak to call one player.)*

## 5. How a team uses it, week to week

**Tiers, not ranks.** Inside the 90–110 band, Combine Juice+ barely orders players (correlation 0.07). The
honest read is three tiers, and the bottom tier is where it's most useful:

In [ ]:
FIG("fig5_tiers.png")

Only **1 of 39** players under 90 became elite movers in games (top 16% at their position), against **20 of 61**
at 110+. Combine juice also predicts **reaching the field**: 88% of players at 110+ logged 50 usable plays,
against 57% of those under 90.

**How many snaps is the Combine worth?** For a rookie, we predict his movement over the rest of the season from
his first *k* NFL plays, with and without his Combine:

In [ ]:
FIG("fig4_snap_takeover.png")

- **Top speed:** game film beats the Combine after **10 plays** (pre-registered **P4**), but the Combine
  keeps adding **+0.08 R²** even at 100 plays. Keep the 40 in the model all season.
- **Agility and burst:** game film wins at once, and adding the Combine doesn't help the in-season read.

Together with section 4, this gives a simple rule for staff. **Early in a season, trust the film for agility.
Across seasons, trust the Combine for agility development.** A rookie whose drill agility outran his game
agility is a candidate to grow into a bigger role in year 2.

## 6. Limits

- **Sample size.** There are about 250 players with both scores, and 30–60 per position. Position-level
  intervals are wide, and a pooled gap smaller than about 0.09 can't be detected reliably.
- **The agility result was a lead before it was a test.** We spotted it in an audit, then wrote it down as
  P1 before the final run. It holds in 2023 (+0.20) and 2024 (+0.29) but **not in 2025** (+0.03); for WR and DB game
  agility, 2025 alone is +0.15 [−0.21, +0.47]. Treat it as
  strong but not settled.
- **Burst is still noisy.** It repeats at only 0.47 from one 40 attempt to the next.
- **Offensive line is a side note.** Combine burst shows up when linemen release downfield on screens
  (+0.36 [+0.15, +0.53] holding the 40, mixed model), but OL speed and agility don't carry over, and we can't
  tie OL movement to pass protection with this data.
- **Football results.** No Combine score predicts them well, and our pre-registered result tests (P2, P3) did
  not pass. Our claim is about movement first, results second.
- **Regular season only.** 96 of the 108 players with no regular-season snap appear in preseason. We did not
  use preseason.
- **Pre-registration.** Five primary tests (P1–P5) were fixed before the final run, and P1, P4 and P5 passed.
  Everything else is exploratory.

**Bottom line.** The Combine's chips mostly re-measure what the stopwatch already knew. The exception is
position-drill agility for receivers and defensive backs. It shows up in games, beyond the 40, and it flags
players likely to grow into it in year 2.

In [ ]:
# Appendix: every number quoted above, recomputed from this run's tables. A failed check means the text needs updating.
import numpy as np
adds = T("tracking_adds.csv"); mm = T("mixed_models.csv"); co = T("carry_over.csv"); pt = T("primary_tests.csv")
tiers = T("tiers.csv"); rf = T("reached_field.csv"); bv = T("burst_validity.csv").set_index("feature")
sf = T("football_results_size_free_pooled.csv"); sz = T("football_results_size_free.csv")
asp = T("asp_combine_vs_game.csv"); uj = T("unused_juice.csv").set_index("metric")
tk = T("snap_takeover.csv"); r2 = T("out_of_class_r2.csv"); summ = json.loads((OUT / "transfer_summary.json").read_text())
pre = json.loads((OUT / "preseason_summary.json").read_text())
cj = T("combine_juice_scores.csv"); gj = T("game_juice_scores.csv")
sc = T("secondary_checks.csv").set_index("id"); spec = T("agility_specificity.csv")
fl = T("agility_floor_sensitivity.csv"); prs = T("agility_pairs.csv")
cw = (OUT / "casework_report.txt").read_text(); olr = T("ol_burst_rescue.csv").set_index("measure")
import re
yr2 = re.search(r"Combine agility ([+-][0-9.]+) \[[^]]*\]   the 40 ([+-][0-9.]+)", cw)

def add(piece, grp, hold="the 40", tgt="Game Juice+"):
    r = adds[(adds.piece.str.startswith(piece)) & (adds.group == grp) & (adds.holding == hold) & (adds.target == tgt)]
    return r.iloc[0]
def mix(grp, out, trait):
    return mm[(mm.group == grp) & (mm.outcome == out) & (mm.trait == trait)].iloc[0]
P = pt.set_index("test"); C = co.set_index("score"); SF = sf.set_index(["pool", "score"])
heath = cj.merge(gj, on="nfl_id", suffixes=("", "_g")).query("display_name == 'Malik Heath'").iloc[0]

checks = {
    "burst d15 vs 40 = 0.20": (bv.loc["d15", "vs_forty"], 0.20),
    "burst split vs 40 = 0.49": (bv.loc["split", "vs_forty"], 0.49),
    "burst split retest = 0.47": (bv.loc["split", "retest"], 0.47),
    "fly10 retest = 0.91": (bv.loc["fly10", "retest"], 0.91),
    "fly10 vs 40 = 0.89": (bv.loc["fly10", "vs_forty"], 0.89),
    "P5 v0 = 0.22": (P.loc["P5", "estimate"], 0.22),
    "P1 agility = 0.17": (P.loc["P1", "estimate"], 0.17),
    "P1 n = 198": (P.loc["P1", "n"], 198),
    "agility WR+DB = 0.30": (add("Agility", "WR + DB").partial_r, 0.30),
    "burst ALL = 0.10": (add("Burst", "ALL").partial_r, 0.10),
    "P2 = 0.15": (P.loc["P2", "estimate"], 0.15),
    "40 vs get-off = 0.37": (summ["decision"]["forty_vs_getoff"], 0.37),
    "carry Juice+ = 0.27": (C.loc["Juice+", "corr_with_game_juice"], 0.27),
    "carry 40 = 0.24": (C.loc["Official 40", "corr_with_game_juice"], 0.24),
    "carry NGS = 0.16": (C.loc["NGS athleticism", "corr_with_game_juice"], 0.16),
    "carry RAS = 0.14": (C.loc["RAS-style", "corr_with_game_juice"], 0.14),
    "carry 40 size-adj = 0.29": (C.loc["Official 40, size-adjusted", "corr_with_game_juice"], 0.29),
    "DB agility transfer = 0.41": (mix("DB", "game agility", "A_trk_plus").transfer, 0.41),
    "WR agility transfer = 0.33": (mix("WR", "game agility", "A_trk_plus").transfer, 0.33),
    "P3 = -0.05": (P.loc["P3", "estimate"], -0.05),
    "WR sep vs weight = -0.42": (sz[(sz.group == "WR") & (sz.outcome == "Separation at throw")].outcome_weight_rho.iloc[0], -0.42),
    "DL get-off vs weight = -0.43": (sz[(sz.group == "DL") & (sz.outcome == "Get-off")].outcome_weight_rho.iloc[0], -0.43),
    "size-free minus NGS = -0.05": (SF.loc[("All blocks", "Juice+ minus NGS"), "estimate"], -0.05),
    "size-free minus RAS = -0.04": (SF.loc[("All blocks", "Juice+ minus RAS-style"), "estimate"], -0.04),
    "size-free minus 40 = -0.01": (SF.loc[("All blocks", "Juice+ minus 40"), "estimate"], -0.01),
    "unused agility (all groups) = 0.26": (uj.loc["game agility", "beta_combine"], 0.26),
    "takeover top speed k = 10": (summ["takeover_k"]["game top speed"], 10),
    "combine adds at 100 = 0.08": (tk[(tk.metric == "game top speed") & (tk.k == 100)].combine_adds.iloc[0], 0.08),
    "inside 90-110 = 0.07": (summ["smaller_checks"]["tiers_inside_90_110"], 0.07),
    "tier <90 elite = 1": (tiers.set_index("tier").loc["<90", "elite_in_games"], 1),
    "tier 110+ elite = 20": (tiers.set_index("tier").loc["110+", "elite_in_games"], 20),
    "reached 110+ = 0.88": (rf.set_index("tier").loc["110+", "share_scored"], 0.884),
    "reached <90 = 0.57": (rf.set_index("tier").loc["<90", "share_scored"], 0.574),
    "preseason-only = 96": (pre["no_regular_season_but_preseason"], 96),
    "Heath 40 = 4.64": (heath.forty, 4.64), "Heath A+ = 119": (heath.A_trk_plus, 119),
    "Heath game agility = 106": (heath.g_agil_plus, 106), "Heath Juice+ = 91": (heath.juice_final, 91),
    "S1 agility -> game agility = 0.33": (sc.loc["S1", "estimate"], 0.33),
    "S1 lower = 0.15": (sc.loc["S1", "ci_low"], 0.15),
    "S2 weighted = 0.34": (sc.loc["S2", "estimate"], 0.34),
    "floor min (mean) = 0.25": (fl[fl.stat == "mean"].r.min(), 0.25),
    "floor max (mean) = 0.34": (fl[fl.stat == "mean"].r.max(), 0.34),
    "year-2 WR+DB agility = 0.34": (float(yr2.group(1)), 0.34),
    "year-2 WR+DB 40 = 0.01": (float(yr2.group(2)), 0.01),
    "pair test = 0.53": (prs.agile_wins_game.mean(), 0.53),
    "OL burst link 1 (mixed) = 0.36": (olr.loc["mixed-model effect, 1+ play", "l1"], 0.36),
    "Heath speed = 86": (heath.T_trk_plus, 86), "Heath plays = 229": (heath.npl, 229),
}
bad = {k: round(float(v), 3) for k, (v, want) in checks.items()
       if abs(float(v) - want) > (0.5 if abs(want) >= 2 else 0.006)}
print(f"{len(checks) - len(bad)} of {len(checks)} quoted numbers match run {RUN['run_id']}.")
if bad:
    print("Update the text for:", bad)